# 🍇 VibeNet Life-Simulator — Dev Launcher

Use this notebook to **start** or **stop** the game server running locally.

- **URL:** [http://localhost:8000](http://localhost:8000)
- **API Docs:** [http://localhost:8000/docs](http://localhost:8000/docs)

> **First time?** Run **Cell 1 (Setup)** once to install all dependencies into this kernel, then run **Cell 2** for the launch panel.

---

## ⚙️  Cell 1 — Setup (run once)

In [ ]:
import sys, subprocess, pathlib

print(f"Kernel Python: {sys.executable}")

# Install all project requirements into this kernel's Python
req_file = pathlib.Path("requirements.txt")
if req_file.exists():
    print("\nInstalling from requirements.txt …")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(req_file)],
        capture_output=True, text=True
    )
    # Show only lines that are meaningful (installed / already satisfied / errors)
    for line in result.stdout.splitlines():
        if any(k in line for k in ("Successfully", "already satisfied", "ERROR", "error")):
            print(" ", line)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-800:])
else:
    print("requirements.txt not found — skipping.")

# Also ensure ipywidgets is available (needed for the launcher UI)
try:
    import ipywidgets  # noqa: F401
    print("\n✅ ipywidgets already installed")
except ImportError:
    print("\nInstalling ipywidgets …")
    subprocess.run([sys.executable, "-m", "pip", "install", "ipywidgets"], check=True)

print("\n✅ Setup complete — run Cell 2 to launch the server.")

---
## ▶️  Cell 2 — Launch panel

In [ ]:
import subprocess
import threading
import time
import sys
import os
import webbrowser
import pathlib
import requests
from IPython.display import display
import ipywidgets as widgets

# ─── Configuration ────────────────────────────────────────────────────────────
HOST       = "127.0.0.1"
PORT       = 8000
APP_MODULE = "backend.main:app"
BASE_URL   = f"http://{HOST}:{PORT}"

# Project root = folder containing this notebook
PROJ_DIR = str(pathlib.Path().resolve())

# Internal state
_server_proc = None
_log_lines   = []

# ─── Widgets ──────────────────────────────────────────────────────────────────
status_label  = widgets.HTML(value='<b style="color:#888">⚪ Server not running</b>')
log_area      = widgets.Textarea(
    value='', placeholder='Server output will appear here…',
    layout=widgets.Layout(width='100%', height='240px'),
    disabled=True
)
btn_start     = widgets.Button(description='▶  Start Server',    button_style='success',
                               layout=widgets.Layout(width='160px'))
btn_stop      = widgets.Button(description='⏹  Stop Server',     button_style='danger',
                               layout=widgets.Layout(width='160px'), disabled=True)
btn_open      = widgets.Button(description='🌐  Open in Browser', button_style='info',
                               layout=widgets.Layout(width='180px'), disabled=True)
btn_clear_log = widgets.Button(description='🧹  Clear Log',       button_style='',
                               layout=widgets.Layout(width='140px'))

controls = widgets.HBox([btn_start, btn_stop, btn_open, btn_clear_log])
panel    = widgets.VBox([status_label, controls, log_area])

# ─── Helpers ──────────────────────────────────────────────────────────────────
def _append_log(line: str):
    _log_lines.append(line)
    if len(_log_lines) > 300:
        _log_lines.pop(0)
    log_area.value = '\n'.join(_log_lines)

def _stream_output(proc):
    """Background thread: pipe server stdout/stderr into the log widget."""
    for raw in proc.stdout:
        _append_log(raw.rstrip('\n'))
    _append_log('[Server process ended]')

def _wait_for_server(timeout: int = 20) -> bool:
    """Poll until the FastAPI server responds or timeout expires."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            r = requests.get(f"{BASE_URL}/api/timeline", timeout=1)
            if r.status_code < 500:
                return True
        except Exception:
            pass
        time.sleep(0.5)
    return False

# ─── Button callbacks ─────────────────────────────────────────────────────────
def on_start(b):
    global _server_proc
    if _server_proc and _server_proc.poll() is None:
        _append_log('[Server is already running]')
        return

    _log_lines.clear()
    log_area.value = ''
    status_label.value = '<b style="color:#f0a500">🟡 Starting…</b>'
    btn_start.disabled = True

    cmd = [
        sys.executable, '-m', 'uvicorn',
        APP_MODULE,
        '--host', HOST,
        '--port', str(PORT),
        '--reload',
    ]
    _append_log(f'$ {" ".join(cmd)}')

    _server_proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd=PROJ_DIR,
    )
    threading.Thread(target=_stream_output, args=(_server_proc,), daemon=True).start()

    def _check_ready():
        if _wait_for_server():
            status_label.value = (
                f'<b style="color:#22c55e">🟢 Running — '
                f'<a href="{BASE_URL}" target="_blank">{BASE_URL}</a></b>'
            )
            btn_stop.disabled = False
            btn_open.disabled = False
            _append_log(f'[Server ready at {BASE_URL}]')
        else:
            status_label.value = '<b style="color:#ef4444">🔴 Failed to start — check the log above</b>'
            btn_start.disabled = False

    threading.Thread(target=_check_ready, daemon=True).start()


def on_stop(b):
    global _server_proc
    if _server_proc and _server_proc.poll() is None:
        _append_log('[Stopping server…]')
        _server_proc.terminate()
        try:
            _server_proc.wait(timeout=5)
        except subprocess.TimeoutExpired:
            _server_proc.kill()
        _append_log('[Server stopped]')
    else:
        _append_log('[No running server to stop]')
    status_label.value = '<b style="color:#888">⚪ Server not running</b>'
    btn_start.disabled = False
    btn_stop.disabled  = True
    btn_open.disabled  = True


def on_open(b):
    webbrowser.open(BASE_URL)


def on_clear_log(b):
    _log_lines.clear()
    log_area.value = ''


btn_start.on_click(on_start)
btn_stop.on_click(on_stop)
btn_open.on_click(on_open)
btn_clear_log.on_click(on_clear_log)

display(panel)

---
## 🔍  Cell 3 — Quick health-check (optional)

Run anytime to confirm the API is reachable.

In [ ]:
import requests

try:
    r = requests.get(f"{BASE_URL}/api/timeline", timeout=3)
    print(f"✅ Server is UP  — HTTP {r.status_code}")
    posts = r.json()
    print(f"   Timeline posts loaded: {len(posts)}")
except requests.ConnectionError:
    print("❌ Server is DOWN (connection refused)")
except Exception as e:
    print(f"⚠️  Unexpected error: {e}")

---
## 🛑  Cell 4 — Emergency kill (if kernel is restarted)

If you restart the Jupyter kernel the widget state is lost but the server process may still be running.
Run this cell to kill any process still listening on port 8000.

In [ ]:
import subprocess, sys

PORT = 8000

if sys.platform == 'win32':
    result = subprocess.run(['netstat', '-ano'], capture_output=True, text=True)
    pids = set()
    for line in result.stdout.splitlines():
        if f':{PORT}' in line and 'LISTENING' in line:
            parts = line.split()
            pids.add(parts[-1])
    if pids:
        for pid in pids:
            subprocess.run(['taskkill', '/PID', pid, '/F'], capture_output=True)
            print(f"Killed PID {pid}")
    else:
        print(f"No process found listening on port {PORT}")
else:
    result = subprocess.run(['fuser', '-k', f'{PORT}/tcp'], capture_output=True, text=True)
    print(result.stdout or result.stderr or f"Done — freed port {PORT}")